## Librerias

In [20]:
import pandas as pd
import statsmodels.api as sm

## Carga datos clean

In [21]:
df_ops = pd.read_csv('C:\\Users\\Usuario\\ProjecteData\\Equip_25\\Data\\clean_data_24-03-2026.csv', parse_dates=['insert_date'])

# inventario activo
df_active = df_ops.loc[df_ops["has_availability"] == True, ["apartment_id","host_id", "room_type", "bedrooms", "bathrooms", "beds", "accommodates", 
                                                            "availability_30", "availability_60", "availability_90", "availability_365", "minimum_nights", 
                                                            "maximum_nights", "city",
                                                            "insert_date", "is_instant_bookable", "price", "price_missing"]].copy()

df_active.info()
df_active.head()

<class 'pandas.core.frame.DataFrame'>
Index: 9116 entries, 0 to 9649
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   apartment_id         9116 non-null   int64         
 1   host_id              9116 non-null   int64         
 2   room_type            9116 non-null   object        
 3   bedrooms             9048 non-null   float64       
 4   bathrooms            9043 non-null   float64       
 5   beds                 9072 non-null   float64       
 6   accommodates         9116 non-null   int64         
 7   availability_30      9116 non-null   int64         
 8   availability_60      9116 non-null   int64         
 9   availability_90      9116 non-null   int64         
 10  availability_365     9116 non-null   int64         
 11  minimum_nights       9116 non-null   int64         
 12  maximum_nights       9116 non-null   int64         
 13  city                 9116 non-null   o

,apartment_id,host_id,room_type,bedrooms,bathrooms,beds,accommodates,availability_30,availability_60,availability_90,availability_365,minimum_nights,maximum_nights,city,insert_date,is_instant_bookable,price,price_missing
0,11964,45553,Private room,1.0,2.0,1.0,2,7,20,40,130,3,365,Malaga,2018-07-31,False,400.0,False
1,21853,83531,Private room,1.0,1.0,1.0,1,0,0,0,162,4,40,Madrid,2020-01-10,False,170.0,False
2,32347,139939,Entire home/apt,2.0,1.0,2.0,4,26,31,31,270,2,120,Sevilla,2019-07-29,True,990.0,False
3,35379,152232,Private room,1.0,2.0,1.0,2,9,23,49,300,2,730,Barcelona,2020-01-10,True,400.0,False
4,35801,153805,Private room,2.0,1.0,5.0,5,0,19,49,312,1,180,Girona,2019-02-19,False,900.0,False


In [22]:
host_counts = df_active.groupby("host_id")["apartment_id"].nunique()

host_counts.describe()
host_counts.value_counts().head(10)

apartment_id
1     6017
2      449
3      142
4       74
6       32
5       32
7       22
9       18
8       12
16       8
Name: count, dtype: int64

#### Definición de métricas
- Disponibilidad: Normalizas horizontes temporales, permite comparabilidad, availability_30 captura demanda reciente
- Capacidad total: bedrooms no captura sofás cama, etc., accommodates es la capacidad real de demanda
- Densidad: listings sobrecargados vs cómodos
- Tamaño categorizado (para interpretación): mejora la lectura de resultados

In [ ]:
# disponibilidad
df_active['availability_30_pct'] = df_active['availability_30'] / 30
df_active['availability_60_pct'] = df_active['availability_60'] / 60
df_active['availability_90_pct'] = df_active['availability_90'] / 90
df_active['availability_365_pct'] = df_active['availability_365'] / 365

available_cols = ["availability_30_pct", "availability_60_pct", "availability_90_pct", "availability_365_pct"]

# habitaciones
df_active['bedrooms'] = df_active['bedrooms'].fillna(0)
# baños
df_active['bathrooms'] = pd.to_numeric(df_active['bathrooms'], errors='coerce')
# camas
df_active['beds'] = df_active['beds'].fillna(0)

# capacidad total
df_active['capacity'] = df_active['accommodates']

# densidad
df_active['bed_density'] = df_active['beds'] / df_active['accommodates']

# tamaño categorizado
df_active['size_segment'] = pd.cut(
    df_active['accommodates'],
    bins=[0,2,4,6,10,20],
    labels=['1-2','3-4','5-6','7-10','10+']
)

# bed density
df_active['bed_density'] = df_active['beds'] / df_active['accommodates']


accommodates
False    9116
Name: count, dtype: int64

## Análisis / visualizaciones
Com afecta el nombre d'habitacions, banys i llits
disponibles a la disponibilitat mitjana dels allotjaments? Difereix entre ciutats?

In [ ]:
# Media simple
df_active.groupby('bedrooms')['availability_30_pct'].mean()

bedrooms
0.0     0.415117
1.0     0.424089
2.0     0.439565
3.0     0.438349
4.0     0.470075
5.0     0.512407
6.0     0.513725
7.0     0.762500
8.0     0.520833
9.0     0.441667
10.0    0.216667
12.0    0.491667
14.0    0.483333
16.0    0.300000
50.0    1.000000
Name: availability_30_pct, dtype: float64

In [42]:
# Por ciudad
df_active.groupby(['city','bedrooms'])['availability_30_pct'].mean()

city       bedrooms
Barcelona  0.0         0.431455
           1.0         0.409491
           2.0         0.399540
           3.0         0.420394
           4.0         0.414634
                         ...   
Valencia   1.0         0.464539
           2.0         0.450483
           3.0         0.381667
           4.0         0.394872
           5.0         0.691667
Name: availability_30_pct, Length: 70, dtype: float64

In [43]:
df_active[['bedrooms','beds','accommodates']].corr()

,bedrooms,beds,accommodates
bedrooms,1.000000,0.774495,0.796162
beds,0.774495,1.000000,0.861774
accommodates,0.796162,0.861774,1.000000


#### Test estadístico
Modelo de regresión lineal múltiple
- Variable dependiente: availability_30_pct
- Variables independientes: bathrooms, bed_density, capacity, city_map, room_type_map

In [49]:
# 1. Variables base
df_model = df_active.copy()

# 3. Seleccionar variables
X = df_model[['accommodates', 'bed_density', 'bathrooms', 'city', 'room_type']]
y = df_model['availability_30_pct']

# 4. Crear dummies
X = pd.get_dummies(X, columns=['city', 'room_type'], drop_first=True)

# 5. Convertir a numérico
X = X.astype(float)

# 6. Limpiar NaNs
df_model_final = pd.concat([X, y], axis=1).dropna()

X = df_model_final.drop(columns=['availability_30_pct'])
y = df_model_final['availability_30_pct']

# 7. Añadir constante
X = sm.add_constant(X)

# 8. Modelo
model = sm.OLS(y, X).fit()

print(model.summary())

                             OLS Regression Results                            
Dep. Variable:     availability_30_pct   R-squared:                       0.014
Model:                             OLS   Adj. R-squared:                  0.012
Method:                  Least Squares   F-statistic:                     9.653
Date:                 Tue, 24 Mar 2026   Prob (F-statistic):           2.36e-20
Time:                         12:48:39   Log-Likelihood:                -4408.6
No. Observations:                 9043   AIC:                             8845.
Df Residuals:                     9029   BIC:                             8945.
Df Model:                           13                                         
Covariance Type:             nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const             

## Resultados / archivos generados

In [ ]:
df_base = df_active.copy()